# UnCLIP: the model behind Dall-E-2 image generation model of Open AI



In this tutorial, we will create and train an UnCLIP model with [TorchDiff](https://loqmansamani.github.io/torchdiff/) diffusion library, a diffusion library built on top of the [PyTorch](https://pytorch.org/) API, which consist of the unclip model (Dall-E-2) implemented based on it main paper [Hierarchical Text-Conditional Image Generation with CLIP Latents](https://arxiv.org/abs/2204.06125). Note that this tutorial will not train a model in real-time; instead, it will demonstrate how to train an UnCLIP model using the TorchDiff API.

<br>
<div align="center">
  <img src="unclip_.png" alt="unclip overview" width="1000"/>
  <br>
  <em>On overview of UnCLIP pipeline from the main paper.</em>
  <br><br>
</div>

## Table of Contents

- [Data Preparation](#data-preparation)
- [Train a  diffusion prior model](#train-diffusion-prior)
- [Train the decoder model](#train-decoder)
- [Train the first upsampler: 64x64 -> 256x256](#train-first-upsampler)
- [Train the second upsampler: 256x256 -> 1024x1024](#train-second-upsampler)
- [Sampling pipeline](#sampling-pipeline)

In [1]:
# Import all necessary libraries and modules for training an UnCLIP model
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset




# Import required classes from the UnCLIP module of TorchDiff

# main diffusion classes needed to apply diffusion
from unclip import VarianceSchedulerUnCLIP, ForwardUnCLIP, ReverseUnCLIP

# classes related to clip models used for embedding images and prompts (texts) into clip latent space
from unclip import CLIPEncoder, CLIPContextProjection, CLIPEmbeddingProjection

# classes related to prior transformer based model of unclip
from unclip import UnCLIPTransformerPrior, TrainUnCLIPPrior

# classes related to decoder model of unclip
from unclip import UnClipDecoder, TrainUnClipDecoder

# classes related to unsampler models
from unclip import UpsamplerUnCLIP, TrainUpsamplerUnCLIP

# the sampler pipe line of unclip
from unclip import SampleUnCLIP

# Import utility functions from the TorchDiff utils module
from utils import NoisePredictor, TextEncoder, Metrics

## Data Preparation

For this tutorial, we will use the [CIFAR-10](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html) dataset from [Torchvision](https://pytorch.org/vision/stable/) with a descriptive caption (mock caption which we will add to each image only for education purposes in this toturial). This dataset consists of low-resolution RBG images, making it suitable for this tutorial.

In [2]:
# Use CIFAR-10 with descriptive captions
class CIFAR10WithCaptions(Dataset):
    def __init__(self, cifar_dataset):
        self.dataset = cifar_dataset
        self.class_names = [
            'airplane', 'automobile', 'bird', 'cat', 'deer',
            'dog', 'frog', 'horse', 'ship', 'truck'
        ]
        # More descriptive templates
        self.templates = [
            "A photo of a {}",
            "An image of a {}",
            "A picture of a {}",
            "This is a {}",
        ]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        class_name = self.class_names[label]
        # Use different templates for variety
        template = self.templates[idx % len(self.templates)]
        caption = template.format(class_name)
        return image, caption


# Updated transforms for CLIP
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 with captions
cifar_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_dataset = CIFAR10WithCaptions(cifar_train)
test_dataset = CIFAR10WithCaptions(cifar_test)

### Using a Subset of the CIFAR10 Dataset

This tutorial will use a small subset of the CIFAR10 dataset for training and validation. Specifically.

In [7]:
# Small subset for testing
train_subset_indices = torch.randperm(len(train_dataset))[:100]
test_subset_indices = torch.randperm(len(test_dataset))[:20]

train_subset = Subset(train_dataset, train_subset_indices)
test_subset = Subset(test_dataset, test_subset_indices)

# DataLoaders
t_loader = DataLoader(train_subset, batch_size=32, shuffle=True, pin_memory=True)
val = DataLoader(test_subset, batch_size=10, shuffle=False, pin_memory=True)

### Considerations for Dataset and Scalability

The subset of data which we chose to train is small and not proper to train a diffusion model. However, the focus here is to demonstrate how to use the TorchDiff API. You can scale up the training dataset or switch to more realistic, high-resolution, RGB images based on your specific needs.

### Train a Diffusion Prior Model

To be able to produce image embeddings from input prompts, we need to train a prior model. in the main paper two different proir models
, an Autoregressive (AR) and a Diffusion prior, are tested, which they found that the second one (diffusion prior) is more efficient and 
producess better results than the autoregressive prior model, so we decided to use the second one (diffusion prior) as part of TorchDiff library. this means
the prior model of unclip model in TorchDiff library is a diffusion based prior.

#### Pipeline of Prior model:

    - encode texts (image captions) and images with CLIP text and image endocer, respectively. this process embedds both images and their captions (z_i, z_t). TorchDiff library uses pre-trained CLIP models available on trainsformers library on hugging-face.
    
    - the embedded images (z_i) are then optionaly feed to a projection network (feed-forward network) to reduce their dimentions to make the algorithm more efficient. the main paper uses PCA (principle componet analysis) to do this but we use a trainable projection layer.
    
    - the output of the previous steps (z_t and z_i_reduced) are then used to train the diffusion based model. the noise predictor model of the prior model is a a transformer architecture, which predict unnoised images from noisy images (in contrast to conventional diffusion models, which predict noise, unclip prior predicts clean image embeddings from noisy image embeddings and text embeddings.

let's first prepare necessary modules used inside prior trainer (TrainUnCLIPPrior)

In [8]:
# variance scheduler of unclip model
h_model = VarianceSchedulerUnCLIP(
    num_steps=1000,
    beta_start=1e-4,
    beta_end=0.02,
    trainable_beta=True,
    beta_method="cosine"
)
# forward and reverse diffusion of prior model
d_model = ForwardUnCLIP(h_model)
r_model = ReverseUnCLIP(h_model)


# clip model 
c_model = CLIPEncoder(model_name="openai/clip-vit-base-patch32")


# embedded-text projection network
tp = CLIPEmbeddingProjection(
    clip_embedding_dim = 512,
    transformer_embedding_dim = 320,
    hidden_dim = 480,
    num_layers = 2,
    dropout_rate = 0.1,
    use_layer_norm = True
)

# embedded-image projection network
ip = CLIPEmbeddingProjection(
    clip_embedding_dim = 512,
    transformer_embedding_dim = 320,
    hidden_dim = 480,
    num_layers = 2,
    dropout_rate = 0.1,
    use_layer_norm = True
)

# trainsformer based prior model
p_model = UnCLIPTransformerPrior(
    forward_diffusion=d_model,
    reverse_diffusion=r_model, # will be used during training
    clip_text_projection=tp,  # used during training instead of PCA in the main paper
    clip_image_projection=ip, # used during training instead of PCA in the main paper
    transformer_embedding_dim = 320,
    num_layers = 12,
    num_attention_heads = 8,
    feedforward_dim = 512,
    max_sequence_length = 2,
    dropout_rate = 0.3
)


# optimizer
opt = torch.optim.AdamW([p for p in p_model.parameters() if p.requires_grad], lr=1e-3)


# loss function 
obj = nn.MSELoss()

In [9]:
# specific trainer of prior
trainer = TrainUnCLIPPrior(
    prior_model = p_model,
    clip_model = c_model,
    train_loader = t_loader,
    optimizer = opt,
    objective = obj,
    val_loader = val,
    max_epochs = 10,
    device = "cuda",
    store_path = "prior_model",
    patience = 3,
    warmup_epochs = 2,
    val_frequency = 3,
    use_ddp = False,
    grad_accumulation_steps = 2,
    log_frequency = 1,
    use_compilation = False,
    embedding_output_range = (-1.0, 1.0),
    reduce_clip_embedding_dim = True,
    transformer_embedding_dim = 320,
    normalize_clip_embeddings = True
)

In [10]:
train_losses, best_val_loss = trainer() 

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.20it/s]


Epoch 1/10 | LR: 1.00e-03 | Train Loss: 0.2659


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.21it/s]


Epoch 2/10 | LR: 1.00e-03 | Train Loss: 0.1643


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.28it/s]


...

Epoch 8/10 | LR: 1.00e-03 | Train Loss: 0.0193


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.80it/s]


Epoch 9/10 | LR: 1.00e-03 | Train Loss: 0.0172 | Val Loss: 0.0643
Checkpoint saved: prior_model/best_model.pth


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.77it/s]

Epoch 10/10 | LR: 1.00e-03 | Train Loss: 0.0151
